> **Historical research track.** This notebook documents the earlier trajectory-anomaly experiment. It is preserved as reproducible evidence and does not define the current SADAR Analyst Console product.

# Phase 2 — Data Validation: Supabase `lemd_*` tables

**Closes:** [issue #12](https://github.com/txema-puch/drone-ai-saturdays/issues/12)
**Spec:** `docs/research/trajectory-anomaly/data-workflow.md`
**Phase in `/ml-lifecycle`:** Phase 2 (Data) — schema, volume, validation, snapshot, hash

**Related artifacts** (where this notebook's outputs land and what it depends on):

- `docs/research/trajectory-anomaly/data-workflow.md` — pipeline workflow, cycle pattern, naming conventions, response playbook
- `docs/research/trajectory-anomaly/lifecycle/02-data.md` — audit narrative; each cycle's snapshot row is appended here
- `docs/research/trajectory-anomaly/lifecycle/manifest.yml` — gate state; each cycle's `dataset_hash` is appended under `gates.data.dataset_hash[]`
- `backend/research/src/sadar_research/trajectory_anomaly/data/opensky.py` — Monica's pipeline + the reference implementations of `calculate_flight_phase` and `distance_to_closest_runway` imported by this notebook for the consistency check
- `backend/research/src/sadar_research/trajectory_anomaly/data/supabase_io.py` — Phase 2 I/O helpers imported by this notebook (`discover_lemd_tables`, `load_table_paginated`, `save_snapshot_parquet`, `compute_file_hash`)

---

## What this notebook is for

This notebook closes Phase 2 of the ML lifecycle for our drone trajectory anomaly detection project. The goal is **not** to build a model or even to do exploratory data analysis (Phase 4). It is to answer one question:

> Is the data Monica uploaded to Supabase real, usable, and enough for what we want to build?

Concretely, that means:
- Confirming the schema matches what Monica's pipeline is supposed to produce
- Auditing the data for nulls, duplicates, and out-of-range values
- Verifying that re-deriving Monica's pipeline-computed columns from the raw inputs gives the same results (catches version skew)
- Locking a hashed local snapshot so future phases can verify they are working with the same data
- Producing a verdict: real / usable / enough — yes or no, with reasons

## How this notebook is organized

The notebook follows a `setup → discovery → snapshot → validate → verdict` arc:

```
  1. Setup                    — imports, env, output dirs
  2. Connect                  — Supabase client
  3. Discover                 — find available lemd_YYYY_MM_DD tables
  4. Snapshot + hash          — pull each day, write parquet, record sha256
  5. Pick a representative    — choose one day for per-day validation
  6. Schema audit             — type, nulls, range, unique count per column
  7. Validation checks        — schema / type / null / dup / range / bbox
  8. Pipeline consistency     — re-derive Monica's columns and compare
  9. Distribution sanity      — histograms (sentinel detection only, not EDA)
 10. Class balance            — operation, flight_phase counts
 11. Volume metrics           — cross-day aggregates for the Enough verdict
 12. Verdict                  — Real / Usable / Enough
 13. Next steps               — write 02-data.md, update manifest, ship
```

## Methodology decisions baked in

All seven Phase 2 methodology decisions were locked during design coaching (see `docs/research/trajectory-anomaly/data-workflow.md`):

1. **Critical columns** (must be non-null): `time, icao24, lat, lon, baroaltitude, flight_id`
2. **FAIL action**: document in `02-data.md` + open GitHub issue + keep broken rows in snapshot
3. **Duplicate detection**: report 3 counts — `(flight_id, time)`, `(flight_id, time, lat, lon)`, full-row
4. **Range bounds**: generous physical bounds (catch *broken*, not *out-of-interest*) + LEMD bbox sanity
5. **Pipeline consistency tolerances**: `velocity_kmh < 1e-6`, `dist_to_runway_m < 1m`, `time_utc < 1s`, `flight_phase` exact
6. **Verdict — Real / Usable**: derived mechanically from check outcomes (≤0.1% noise tolerance)
7. **Verdict — Enough**: 4 buckets (NOT_YET / SOFT_DEV / CONDITIONAL / PASS) on trajectory count + day count

## Prerequisites

Before running:
1. `.env` at repo root with credentials for the cycle you want to validate. The notebook uses a **multi-account scheme** — set `ACCOUNT_SLUG` in cell 1 (`cycle1`, `cycle2`, ...) and provide matching `SUPABASE_URL_<SLUG>` / `SUPABASE_KEY_<SLUG>` in `.env`. Unsuffixed `SUPABASE_URL` / `SUPABASE_KEY` and `OPENSKY_USERNAME` / `OPENSKY_PASSWORD` are also required because `sadar_research.trajectory_anomaly.data.config` evaluates `Settings()` at module load time; OpenSky values can be placeholders because this notebook does not call Trino.
2. From the repo root: `uv sync --project backend/research --extra data --extra notebooks`.
3. Select the resulting `backend/.venv/bin/python` interpreter as this notebook's kernel.

Run cells **top to bottom**. Each section is independent enough to re-run, but the order matters for assignments (e.g., cell 4 sets `snapshots`, cell 5 reads it).

## How to read this notebook (the pedagogy)

Every section has a markdown cell that explains:
- *What this cell does* — the mechanical operation
- *Why it matters* — what problem it solves in the lifecycle
- *What to look for* — how to read the output, what's signal vs noise
- *What to do next* — how outcomes feed into 02-data.md and the verdict

If you only want a quick green-light run, scan the headers and the verdict in section 12. If you're learning the validation pattern (or writing the team's writeup), read every markdown cell.

All code-cell variables are kept around for re-use — `df`, `schema_df`, `snapshots`, `result_7a` … `result_8`, etc. — so you can poke at them in additional cells you add at the end.

## 1 — Setup

**What this cell does.** Loads imports, reads the `.env` file, makes the `backend/` package importable, and creates the directories where snapshots and figures will land.

**Why it matters.** Several non-obvious ordering choices live in this cell — getting them wrong produces opaque errors later:

- **`.env` is loaded *before* `from sadar_research.trajectory_anomaly.data.opensky import ...`.** Monica's `backend/research/src/sadar_research/trajectory_anomaly/data/opensky.py` does `from sadar_research.trajectory_anomaly.data.config import settings`, and `data/config.py` evaluates `Settings()` at module load time. `Settings()` reads env vars and raises `ValidationError` if any of `OPENSKY_USERNAME`, `OPENSKY_PASSWORD`, `SUPABASE_URL`, `SUPABASE_KEY` is missing. So if we import before loading `.env`, the import explodes on a teammate machine that has only Supabase creds in their shell.
- **`sys.path.insert(0, REPO_ROOT / "backend" / "research" / "src")`** lets us import `sadar_research.trajectory_anomaly.data.supabase_io` and `sadar_research.trajectory_anomaly.data.opensky` directly from the research package checkout.
- **`mkdir(parents=True, exist_ok=True)`** for both output dirs means re-running the notebook is idempotent — the directories exist if they already did, and don't crash if they don't.

**What to look for in the output.**
- `Repo root: ...` should end in `drone-ai-saturdays`. If it doesn't, the `Path.cwd().parent` heuristic broke and the notebook is being run from somewhere unexpected.
- The `assert SUPABASE_URL and SUPABASE_KEY` line raises `AssertionError` if the env vars are missing or empty. If you see that, your `.env` file isn't loaded — check the file exists at the repo root and contains the keys.
- `Setup complete.` with no exception means everything is wired up. We can connect.

In [ ]:
import os
import sys
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

# Resolve the repository root from any supported notebook working directory
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "backend" / "research" / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
print(f"Repo root: {REPO_ROOT}")

# Load .env BEFORE importing backend modules — they evaluate Settings at import time
load_dotenv(REPO_ROOT / ".env")

# Cycle selector — set the slug of the Supabase account to validate this run.
# Credentials live in .env as SUPABASE_URL_<SLUG> / SUPABASE_KEY_<SLUG>.
# Backward-compat: unsuffixed SUPABASE_URL / SUPABASE_KEY work as fallback.
ACCOUNT_SLUG = "cycle2"  # change per cycle: cycle1, cycle2, cycle3, ...

# Supabase table name for this cycle. Workflow doc allows any single-table
# name (lemd_<suffix>); Monica tells you the name when she signals a batch
# is ready. Set to None to fall back to cycle 1's legacy date-probing path
# (lemd_YYYY_MM_DD) — only used to re-run cycle 1's audit.
TABLE_NAME = "lemd_2026"

slug_upper = ACCOUNT_SLUG.upper()
SUPABASE_URL = os.environ.get(f"SUPABASE_URL_{slug_upper}") or os.environ.get("SUPABASE_URL")
SUPABASE_KEY = os.environ.get(f"SUPABASE_KEY_{slug_upper}") or os.environ.get("SUPABASE_KEY")
assert SUPABASE_URL and SUPABASE_KEY, (
    f"Missing SUPABASE_URL_{slug_upper} / SUPABASE_KEY_{slug_upper} in .env "
    f"(also no unsuffixed fallback found)"
)
print(f"Validating Supabase account: {ACCOUNT_SLUG}")
print(f"Target table: {TABLE_NAME or '(legacy date-probe path)'}")

# Make backend importable for the consistency-check helpers
sys.path.insert(0, str(REPO_ROOT / "backend" / "research" / "src"))
from sadar_research.trajectory_anomaly.data.opensky import calculate_flight_phase, distance_to_closest_runway
from sadar_research.trajectory_anomaly.data.supabase_io import (
    discover_lemd_tables,
    load_table_paginated,
    save_snapshot_parquet,
    compute_file_hash,
)

# Ensure output directories exist
(REPO_ROOT / "data" / "raw").mkdir(parents=True, exist_ok=True)
FIG_DIR = REPO_ROOT / "docs" / "research" / "trajectory-anomaly" / "figures" / "02-data"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete.")

## 2 — Connect to Supabase

**What this cell does.** Instantiates a Supabase Python client using the URL and key loaded from `.env`. The client is what every subsequent cell uses to talk to the database.

**Why it matters.** `create_client` does *not* perform a network round-trip to validate credentials. It just constructs an object. The first real network call happens later (when we run a `.select(...).execute()` query). That means: a bad URL or key won't fail here — it'll fail in cell 3 with a more confusing error.

**What to look for.**
- A bare `Supabase client created.` with no exception is all we expect.
- If the cell ever DOES fail here, it's a malformed URL (bad string, missing https://) or an import-time error inside the `supabase` library — neither common.

**Common failure modes you may see in cell 3 instead of here.**
- `401 Unauthorized` → `SUPABASE_KEY` is wrong (typo, expired, or it's the anon key on a project requiring service-role)
- `Could not resolve host` → `SUPABASE_URL` is wrong (typo, missing https, wrong project)

In [3]:
client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Supabase client created.")

Supabase client created.


## 3 — Resolve the Supabase table to validate

**What this cell does.** Picks which Supabase table the audit will pull from. Two paths:

1. **Explicit `TABLE_NAME` path (default, cycle 2 onward).** Set `TABLE_NAME` in cell 1 to the name Monica gave you for this cycle. The cell verifies the table is reachable with a single-row `select`, then sets `available = [TABLE_NAME]` so the rest of the notebook treats it as the only batch.
2. **Legacy date-probe fallback (cycle 1 only).** Set `TABLE_NAME = None` in cell 1. The cell then probes a date range, asking Supabase for one row from each candidate `lemd_YYYY_MM_DD` name in `[PROBE_START, PROBE_END]`. This path exists so cycle 1's audit can be re-run unchanged; never use it for new cycles.

**Why two paths.** Cycle 1 landed on an accidental per-day naming convention (one Supabase table per day, named `lemd_YYYY_MM_DD`). The cycle 1 audit flagged this as misleading — a table named `lemd_2025_03_10` actually held 5 days of data. The workflow doc (`docs/research/trajectory-anomaly/data-workflow.md`) explicitly states the Supabase table name is a query target, not a description of contents. From cycle 2 onward, Monica picks any single name (e.g., `lemd_2026`) and tells the team; the notebook takes the name as input rather than guessing it.

**Why not just delete the probe path.** It costs ~10 lines of code and keeps cycle 1's audit reproducible. If anyone ever needs to re-validate cycle 1, flip `TABLE_NAME` to `None` in cell 1 and the probe path takes over.

**What to look for.**
- The `Using explicit TABLE_NAME: <name>` line tells you the audit is using the modern path.
- `Available tables: 1` followed by the table name confirms it was reached.

**What to do if you see an error.**
- `RuntimeError: Table <name> not reachable` → confirm the name with Monica, and confirm `SUPABASE_URL_<SLUG>` / `SUPABASE_KEY_<SLUG>` in `.env` point at the correct project (open the Supabase web UI for that project and check the table is listed there).
- For cycle 1 audit replays only: set `TABLE_NAME = None` in cell 1 to use the legacy probe path.

In [ ]:
# Resolve which Supabase table to validate.
# If TABLE_NAME is set in cell 1, use it directly (workflow doc convention
# from cycle 2 onward — single working table per cycle).
# Else fall back to date-based probing (cycle 1 legacy: one table per day).
if TABLE_NAME:
    print(f"Using explicit TABLE_NAME: {TABLE_NAME}")
    try:
        client.table(TABLE_NAME).select("*").limit(1).execute()
    except Exception as exc:
        raise RuntimeError(
            f"Table {TABLE_NAME!r} not reachable: {exc}. Check the name "
            f"with Monica and confirm SUPABASE_URL_{slug_upper} / "
            f"SUPABASE_KEY_{slug_upper} point at the right project."
        ) from exc
    available = [TABLE_NAME]
else:
    PROBE_START = date(2025, 1, 1)
    PROBE_END = date(2025, 12, 31)
    print(f"Probing {(PROBE_END - PROBE_START).days + 1} candidate tables in [{PROBE_START}, {PROBE_END}]...")
    available = discover_lemd_tables(client, PROBE_START, PROBE_END)

print(f"\nAvailable tables: {len(available)}")
for name in available:
    print(f"  {name}")

if not available:
    raise RuntimeError(
        "No lemd_* tables found. If using explicit TABLE_NAME, check the "
        "name with Monica. If using the legacy probe path "
        "(TABLE_NAME=None), widen PROBE_START / PROBE_END or re-check "
        "Supabase credentials."
    )

                                                          
  ## 4 — Pull snapshots, save to parquet, compute hashes
                                                                                                                                                       
  **What this cell does.** For each Supabase table discovered in cell 3, fetches every row (paginated through Supabase's 1000-row default page size),  
  derives a descriptive snapshot name from the data's own `time_utc` range, writes the rows to a parquet file under `data/raw/`, and computes a sha256 
  of that file. Builds a `snapshots` dict that the rest of the notebook uses.                                                                          
                                                            
  **Why it matters.**

  - **Reproducibility.** Phase 6 (training) needs to know exactly *which version* of the data was used. The hash is the contract: any later phase loads
   the parquet, hashes it, compares to the manifest. If they match, same data. If not, something changed.
  - **Speed.** Re-running the rest of the notebook (and Phases 4, 5, 6) reads from local parquet, which is ~50× faster than re-pulling from Supabase.  
  - **Backup.** The Supabase table is transient (gets truncated each cycle when ~500 MB limit approaches). The local parquets are the durable record.  
  Without them, the data is gone after Monica's next truncate.                                                                                         
                                                                                                                                                       
  **Why naming derives from the data, not the table.** The Supabase table is a working store that turns over with each cycle, so its name doesn't describe its current contents. The parquet name encodes the actual date range (`<startYYYYMMDD>_to_<endYYYYMMDD>`) computed from `time_utc.min()` / `time_utc.max()`.That makes every parquet self-describing regardless of what the Supabase table was named or what it contains right now. Naming convention spec: `docs/research/trajectory-anomaly/data-workflow.md`.
  

  **Why parquet (not CSV / JSON).** Parquet preserves dtypes, compresses ~5-10× smaller than CSV, reads faster. The cost is one extra dependency (`pyarrow`).

  **What to look for in the output.**                       

  - One block per discovered table, showing rows, columns, parquet size, derived snapshot name, and sha256 hex.
  - The first line of each block should look like `Loading lemd_<table>... 1,834,084 rows, 21 columns`. If columns is *not* 21, schema has drifted upstream — investigate before continuing.
  - The derived parquet name should match the data's actual date range. Mismatch (e.g., parquet named `lemd_20250310_to_20250314` but containing 2026 data) means `time_utc` was malformed.
  - The size_mb scales with data volume; LEMD typically produces ~30-80 MB compressed per day's worth of data.
  - The 'Snapshot summary' at the bottom is the parquet-name → sha256 mapping you'll paste into `manifest.yml > gates.data.dataset_hash`.

  **Idempotency.** This cell overwrites existing parquet files. Re-running with the same data should produce the same hash. If you re-run and the hash differs without the data changing, something non-deterministic is happening — usually a `pyarrow` version change. (The dev container pins pyarrow to keep this stable.)

  **Critical timing reminder.** Do NOT confirm to Monica that she can truncate the Supabase table until *after* this cell completes successfully and   
  you've verified the parquet hash. If Monica truncates before the snapshot is on disk, the data is gone forever.

In [ ]:
from datetime import date                                                                                                                            
                                                            
snapshots = {}  # parquet_basename -> {path, supabase_table, rows, columns, size_mb, hash_sha256}                                                    
   
for table_name in available:                                                                                                                         
    print(f"\nLoading {table_name}...")                   
    df_t = load_table_paginated(client, table_name)                                                                                                  
    print(f"  {len(df_t):,} rows, {len(df_t.columns)} columns")                                                                                      
                                                                                                                                                       
    # Derive descriptive snapshot name from the data's own time_utc range,                                                                           
    # not from the Supabase table name. The table name is transient
    # (truncated each cycle); the data's date range is what matters.                                                                                 
    times = pd.to_datetime(df_t["time_utc"], utc=True, errors="coerce").dropna()                                                                     
    start = times.min().date().strftime("%Y%m%d")                                                                                                    
    end = times.max().date().strftime("%Y%m%d")                                                                                                      
    snapshot_date = date.today().isoformat()                                                                                                         
    snapshot_name = f"lemd_{start}_to_{end}__snapshot_{snapshot_date}"                                                                               
                                                                                                                                                       
    parquet_path = REPO_ROOT / "data" / "raw" / f"{snapshot_name}.parquet"                                                                           
    save_snapshot_parquet(df_t, parquet_path)                                                                                                        
    digest = compute_file_hash(parquet_path)                                                                                                         
                                                            
    snapshots[snapshot_name] = {                                                                                                                     
        "supabase_table": table_name,                     
        "path": str(parquet_path.relative_to(REPO_ROOT)),                                                                                            
        "rows": int(len(df_t)),
        "columns": int(len(df_t.columns)),                                                                                                           
        "size_mb": round(parquet_path.stat().st_size / 1024 / 1024, 2),                                                                              
        "hash_sha256": digest,
    }                                                                                                                                                
    print(f"  data range:    {start} → {end}")            
    print(f"  -> {parquet_path.name} ({snapshots[snapshot_name]['size_mb']} MB)")                                                                    
    print(f"  sha256: {digest}")                                                                                                                     
                                                                                                                                                       
print("\n" + "=" * 70)                                                                                                                               
print("Snapshot summary (paste into manifest.yml > gates.data.dataset_hash):")                                                                       
print("=" * 70)                                                                                                                                      
for name, info in snapshots.items():
    print(f"  {name}: {info['hash_sha256']}")


Loading lemd_2025_03_10...
  lemd_2025_03_10: fetched 1,000 rows...
  lemd_2025_03_10: fetched 2,000 rows...
  lemd_2025_03_10: fetched 3,000 rows...
  lemd_2025_03_10: fetched 4,000 rows...
  lemd_2025_03_10: fetched 5,000 rows...
  lemd_2025_03_10: fetched 6,000 rows...
  lemd_2025_03_10: fetched 7,000 rows...
  lemd_2025_03_10: fetched 8,000 rows...
  lemd_2025_03_10: fetched 9,000 rows...
  lemd_2025_03_10: fetched 10,000 rows...
  lemd_2025_03_10: fetched 11,000 rows...
  lemd_2025_03_10: fetched 12,000 rows...
  lemd_2025_03_10: fetched 13,000 rows...
  lemd_2025_03_10: fetched 14,000 rows...
  lemd_2025_03_10: fetched 15,000 rows...
  lemd_2025_03_10: fetched 16,000 rows...
  lemd_2025_03_10: fetched 17,000 rows...
  lemd_2025_03_10: fetched 18,000 rows...
  lemd_2025_03_10: fetched 19,000 rows...
  lemd_2025_03_10: fetched 20,000 rows...
  lemd_2025_03_10: fetched 21,000 rows...
  lemd_2025_03_10: fetched 22,000 rows...
  lemd_2025_03_10: fetched 23,000 rows...
  lemd_2025_03_

KeyboardInterrupt: 

## 5 — Pick a representative table for per-day validation

**What this cell does.** Selects the first available table as `TABLE`, loads its parquet snapshot into `df`. The validation cells in section 7-10 operate on `df` (one day's worth of data).

**Why one day at a time.** Per-day validation tests are about *whether each day's data is internally sound* — schema match, type sanity, null check, duplicates, range bounds, and pipeline consistency are all per-day questions. Cross-day questions (do we have enough days? is the total volume sufficient?) belong to section 11 (volume metrics) and feed the **Enough** verdict.

**Switching the day.** To validate a different day, change `TABLE = available[N]` for some other index `N` and re-run sections 6 through 10. Validation outcomes should be consistent across days; if they differ between days, that's itself a finding worth reporting.

**What to look for.**
- `Shape: (N, 21)` — N is the row count for this day. The 21 is the column count. If it's not 21, schema has drifted; investigate before validating further.
- The columns list should match Monica's pipeline output. If you see unexpected names or missing names, the upstream pipeline has changed.

In [22]:
# Pick the representative snapshot for per-batch validation.                                                                                         
# Tries the discovery-flow result first; falls back to the most recent                                                                               
# parquet on disk if the kernel was started without running cells 3-4.                                                                               
                                                                                                                                                       
if "snapshots" in globals() and snapshots:                                                                                                           
    SNAPSHOT_NAME = next(iter(snapshots))                                                                                                            
    TABLE = snapshots[SNAPSHOT_NAME]["supabase_table"]                                                                                               
    parquet_path = REPO_ROOT / "data" / "raw" / f"{SNAPSHOT_NAME}.parquet"                                                                           
else:                                                                                                                                                
    # Restart scenario — find the most recent snapshot on disk.                                                                                      
    candidates = sorted(                                                                                                                             
        (REPO_ROOT / "data" / "raw").glob("lemd_*__snapshot_*.parquet")
    )                                                                                                                                                
    if not candidates:                                    
        raise RuntimeError(                                                                                                                          
            "No snapshot found. Either run cells 3-4 (discovery + pull), "
            "or rename your existing parquet to follow the convention: "                                                                             
            "lemd_<startYYYYMMDD>_to_<endYYYYMMDD>__snapshot_<YYYY-MM-DD>.parquet"                                                                   
        )                                                                                                                                            
    parquet_path = candidates[-1]                                                                                                                    
    SNAPSHOT_NAME = parquet_path.stem                                                                                                                
    TABLE = None  # Supabase table name unknown in restart flow                                                                                      
                                                                                                                                                       
df = pd.read_parquet(parquet_path, engine="pyarrow")                                                                                                 
print(f"Loaded snapshot: {SNAPSHOT_NAME}")                                                                                                           
print(f"  Path:          {parquet_path.relative_to(REPO_ROOT)}")                                                                                     
print(f"  Supabase tbl:  {TABLE if TABLE else '(unknown — restart flow)'}")                                                                          
print(f"  Shape:         {df.shape}")                                                                                                                
print(f"  Columns:       {list(df.columns)}")

Loaded snapshot: lemd_20260310_to_20260314__snapshot_2026-05-11
  Path:          data/raw/lemd_20260310_to_20260314__snapshot_2026-05-11.parquet
  Supabase tbl:  lemd_2026
  Shape:         (1774859, 21)
  Columns:       ['time', 'icao24', 'lat', 'lon', 'baroaltitude', 'geoaltitude', 'velocity', 'heading', 'vertrate', 'callsign', 'onground', 'squawk', 'alert', 'spi', 'lastcontact', 'flight_id', 'operation', 'time_utc', 'velocity_kmh', 'dist_to_runway_m', 'flight_phase']


## 6 — Schema audit

**What this cell does.** Builds a one-row-per-column reference table (`schema_df`) summarizing each column's pandas type, null count and percentage, unique-value count, and (for numerics) min / max / mean. Each column is also marked as **raw** (15 columns from OpenSky) or **derived** (6 columns added by Monica's pipeline) per the data lineage in the design doc.

**Why it matters.** This is the first 'look the data in the eye' moment. Many subtle data issues surface from a careful read of this table — issues that would otherwise hide until they bite a downstream phase. Looking systematically at every column up front is far cheaper than chasing a Phase 6 mystery (`'why is my LSTM not converging?'`) back to a column dtype problem in Phase 2.

**What each column of `schema_df` is for.**
- **`column`** — the column name from the parquet file.
- **`kind`** — `raw` (came directly from OpenSky's `state_vectors_data4`) or `derived` (computed by Monica's `build_master_table`). Mismatches with the design doc's expected lineage should be investigated.
- **`dtype`** — the pandas type after parquet round-trip. Watch for:
  - `object` on columns we expect numeric → strings snuck in (often a sentinel encoded as text)
  - `int64` on columns that should be float → loss of NaN representation
  - `object` for booleans → JSON serialization issue (`'true'` vs `True`)
  - `string[python]` vs `object` for strings — both are common, neither is wrong, but mixing within one source is suspicious.
- **`nulls` / `null_pct`** — null count and percentage. Critical-column nulls are FAILed in cell 7c. Non-critical nulls are just reported here so we know what to expect.
- **`n_unique`** — distinguishes categorical-like columns (e.g., `flight_phase` should have ~6 distinct values) from continuous numerics (millions of values).
- **`min` / `max` / `mean`** (numeric only) — informal range check before the formal one in cell 7e. Spot sentinels here: a `min` of `-9999` or `-1` on altitude usually means 'GPS error' encoded as a sentinel value.

**What to look for.**
- All 21 expected columns present, in `kind` of `raw` (15) + `derived` (6).
- `dtype` looks plausible for each column (e.g., `time` is `int64`, `lat`/`lon`/`altitude` are `float64`, `icao24`/`callsign` are `object` or `string`).
- `null_pct` is 0% for the critical columns and reasonable for non-critical ones. ADS-B has known reasons for `callsign` and `squawk` to occasionally be null; >50% nulls would be suspicious.
- `n_unique` for `flight_phase` ≈ 6 (the rule's output classes). If it's hundreds or thousands, something is wrong with the rule.
- `min` and `max` are within roughly the expected physical ranges. Anything wildly off (negative altitudes, > 1000 m/s velocities) flags an issue.

**What this gives you for later.** `schema_df` is referenced by cell 7c (null check), cell 7e (range check), and is one of the first things to copy into `02-data.md`'s 'Schema (data dictionary)' section. Keep it around.

In [23]:
RAW_COLUMNS = {
    "time", "icao24", "lat", "lon", "baroaltitude", "geoaltitude",
    "velocity", "heading", "vertrate", "callsign", "onground", "squawk",
    "alert", "spi", "lastcontact",
}
DERIVED_COLUMNS = {
    "flight_id", "operation", "time_utc",
    "velocity_kmh", "dist_to_runway_m", "flight_phase",
}

def column_kind(col: str) -> str:
    if col in RAW_COLUMNS:
        return "raw"
    if col in DERIVED_COLUMNS:
        return "derived"
    return "unknown"

rows = []
for col in df.columns:
    s = df[col]
    entry = {
        "column": col,
        "kind": column_kind(col),
        "dtype": str(s.dtype),
        "nulls": int(s.isna().sum()),
        "null_pct": round(100 * s.isna().sum() / len(df), 2),
        "n_unique": int(s.nunique(dropna=True)),
    }
    if pd.api.types.is_numeric_dtype(s):
        non_null = s.dropna()
        if len(non_null):
            entry["min"] = float(non_null.min())
            entry["max"] = float(non_null.max())
            entry["mean"] = round(float(non_null.mean()), 4)
    rows.append(entry)

schema_df = pd.DataFrame(rows)
schema_df

,column,kind,dtype,nulls,null_pct,n_unique,min,max,mean
0,time,raw,int64,0,0.00,312292,1.773117e+09,1.773532e+09,1.773326e+09
1,icao24,raw,str,0,0.00,350,NaN,NaN,NaN
2,lat,raw,float64,0,0.00,142717,3.864656e+01,4.231883e+01,4.046690e+01
3,lon,raw,float64,0,0.00,212927,-5.961519e+00,-1.156860e+00,-3.419000e+00
4,baroaltitude,raw,float64,3,0.00,1651,4.724400e+02,1.978152e+04,5.394774e+03
5,geoaltitude,raw,float64,109,0.01,1637,6.248400e+02,1.998726e+04,5.530575e+03
6,velocity,raw,float64,14,0.00,55077,0.000000e+00,2.953867e+02,1.731042e+02
7,heading,raw,float64,14,0.00,232080,0.000000e+00,3.598776e+02,1.819418e+02
8,vertrate,raw,float64,14,0.00,212,-9.265920e+01,3.673856e+01,-6.892000e-01
9,callsign,raw,str,0,0.00,642,NaN,NaN,NaN


## 7 — Validation checks (overview)

Five checks, each producing a clear PASS / FAIL / REVIEW outcome. Each is tied to one of the locked Phase 2 design decisions and produces a `result_*` variable used by the verdict in section 12.

| # | Check | Tied to decision | What it catches |
|---|---|---|---|
| 7a | Schema match | (implicit) | Columns missing or unexpected — wrong dataset, upstream schema change |
| 7b | Type sanity | (implicit) | Dtype drift — strings where numbers expected, etc. |
| 7c | Critical nulls | Decision 1 | Structurally broken rows (`time`, `icao24`, `lat`, `lon`, `baroaltitude`, `flight_id` non-null) |
| 7d | Duplicate detection | Decision 3 | Pipeline re-insert bugs and multi-receiver edge cases |
| 7e | Range bounds + LEMD bbox | Decision 4 | Sentinel values, broken data, geography mismatches |

Each check writes its outcome to a variable (`result_7a`, ..., `result_7e`). The verdict cell (12) reads these to compute Real / Usable.

**On FAIL action (Decision 2).** When a check FAILs:
1. The notebook prints the offending rows (sample) and counts.
2. Document the finding in `02-data.md > Known issues`.
3. Open a GitHub issue tagged `[Bug]` and assigned to the team (Monica + everyone) — see https://github.com/txema-puch/drone-ai-saturdays/issues/new
4. **Keep the broken rows in the snapshot** so the snapshot mirrors what's in Supabase. Phase 3 (Preprocess) decides whether to drop / impute / flag during preprocessing.

### 7a — Schema match

**What this cell does.** Compares the actual column set against the 21 expected columns (15 raw + 6 derived). Prints any missing or extra columns and a PASS / FAIL outcome.

**Why it matters.** Schema match is the cheapest way to catch upstream pipeline drift. If Monica updated `build_master_table` to add or remove a column without telling us, this is where we find out. It's also the first line of defense against 'wrong dataset' — if you accidentally point at a different Supabase project, the schema almost certainly won't match.

**What to look for.**
- `Missing: none` and `Extra: none` → PASS. The schema is what we expect.
- `Missing: ['xxx']` → Monica's pipeline used to produce `xxx` but no longer does. Open an issue, ask Monica before proceeding.
- `Extra: ['yyy']` → Monica's pipeline now produces `yyy` that we didn't account for in the design doc. Update `RAW_COLUMNS` / `DERIVED_COLUMNS` and reconcile.

**What FAIL triggers.** Stop the validation here. The data isn't what we designed for, and continuing would produce misleading downstream check results.

In [24]:
EXPECTED_COLUMNS = sorted(RAW_COLUMNS | DERIVED_COLUMNS)
actual = set(df.columns)
expected = set(EXPECTED_COLUMNS)
missing = expected - actual
extra = actual - expected

print(f"Expected columns: {len(expected)}")
print(f"Actual columns:   {len(actual)}")
print(f"Missing:          {sorted(missing) if missing else 'none'}")
print(f"Extra:            {sorted(extra) if extra else 'none'}")
result_7a = "PASS" if not missing and not extra else "FAIL"
print(f"\nResult: {result_7a}")

Expected columns: 21
Actual columns:   21
Missing:          none
Extra:            none

Result: PASS


### 7b — Type sanity

**What this cell does.** Checks that each numeric column has a numeric dtype and each boolean column has a bool dtype. Reports any mismatches and a PASS / REVIEW outcome.

**Why it matters.** Type drift is a silent killer. Pandas can store integers as `object` (when nulls are present and pyarrow defaulted to nullable), bools as `int64` (`0`/`1` → `False`/`True` was lost), or floats as `object` (when a sentinel like `'NaN'` was a string). Each of these breaks a downstream operation in a different way:
- `object` numerics → comparison operators and arithmetic silently fail
- `int64` bools → `(df['onground'] & ...)` does bitwise instead of logical
- `object` floats → `.mean()` raises

Catching them at Phase 2 saves hours of `'why does my filter return 0 rows'` debugging in Phase 3+.

**What to look for.**
- `All numeric/bool columns have correct types.` → PASS. Move on.
- `Type issues:` listing one or more columns → REVIEW NEEDED. Look at the `sample` value to diagnose. Common cases:
  - `expected numeric, got object; sample=['12.5', '13.7']` → strings instead of floats. Caused by Supabase JSON returning floats as strings (rare; would point to a serialization issue).
  - `expected bool, got object; sample=['true', 'false']` → boolean strings. Coerce with `df['col'].map({'true': True, 'false': False})` and check for sentinel `'unknown'`.
  - `expected bool, got int64; sample=[0, 1]` → integers used as bools. Treat carefully if you use bitwise operators downstream.

**What REVIEW triggers.** Type issues don't necessarily mean the data is unusable — most can be fixed with a `.astype()` or coercion in Phase 3. Document the issues in `02-data.md`'s 'Known issues' section so Phase 3 picks them up.

In [25]:
NUMERIC_COLS = [
    "time", "lat", "lon", "baroaltitude", "geoaltitude",
    "velocity", "heading", "vertrate", "lastcontact",
    "velocity_kmh", "dist_to_runway_m",
]
BOOL_COLS = ["onground", "alert", "spi"]

issues = []
for col in NUMERIC_COLS:
    if col in df.columns and not pd.api.types.is_numeric_dtype(df[col]):
        sample = df[col].dropna().head(3).tolist()
        issues.append(f"  {col}: expected numeric, got {df[col].dtype}; sample={sample}")
for col in BOOL_COLS:
    if col in df.columns and not pd.api.types.is_bool_dtype(df[col]):
        sample = df[col].dropna().head(3).tolist()
        issues.append(f"  {col}: expected bool, got {df[col].dtype}; sample={sample}")

if issues:
    print("Type issues:")
    for i in issues:
        print(i)
    result_7b = f"REVIEW NEEDED ({len(issues)} issues)"
else:
    print("All numeric/bool columns have correct types.")
    result_7b = "PASS"
print(f"\nResult: {result_7b}")

All numeric/bool columns have correct types.

Result: PASS


### 7c — Critical columns null check

**What this cell does.** Verifies the six 'critical' columns (locked during Phase 2 design as Decision 1) have *zero* nulls. Other columns can have nulls and we just report the rate.

**Why these six are critical.** A 'critical' column is one whose nullity makes the row structurally broken — not just less-useful, but literally non-interpretable:

- **`time`** — without it, we don't know when this state vector happened. Can't order, can't bin, can't analyze.
- **`icao24`** — without it, we don't know which aircraft. Even though the LSTM doesn't use it as a feature, downstream identity gate work and the `flight_id` derivation depend on it. A null icao24 is also a smoke signal that something is wrong upstream — Monica's Trino query enforces `WHERE icao24 = ?`, so it shouldn't ever be null.
- **`lat` / `lon`** — without position, the row is meaningless for anomaly detection. The whole project rests on trajectories.
- **`baroaltitude`** — the primary altitude signal. We need at least one altitude per state vector to model normal flight.
- **`flight_id`** — the structural anchor for grouping rows into trajectories. Without it, every row is an island.

**Why velocity, heading, vertrate are NOT critical here.** They're required as ML input features (Phase 5 / 6 concern), but their nullity doesn't make the *row* structurally broken. ADS-B has known reasons for occasional missing speed or heading (transponder limitations, edge of receiver range). Phase 5 decides whether to interpolate, drop, or flag. Phase 2 just records the rate.

**What to look for.**
- `[ok]` next to every critical column → PASS.
- `[FAIL]` on any → critical column has nulls. The data is structurally broken in those rows. **This is a real problem and should never happen with Monica's current pipeline.** It indicates either a Supabase corruption or an upstream pipeline bug.

**What FAIL triggers (Decision 2).**
1. Print sample of offending rows (uncomment the lines below).
2. Document in `02-data.md > Known issues`.
3. Open `[Bug]` GitHub issue tagging the team.
4. Keep the broken rows in the snapshot. Phase 3 decides what to do with them.

In [26]:
# Decision 1: critical columns (Phase 2 design)
# These six are 'must be non-null for the row to be valid'.
# velocity, heading, vertrate are required for ML training (Phase 5/6 concern)
# but their nullity does not break the row's structural integrity.
CRITICAL = ["time", "icao24", "lat", "lon", "baroaltitude", "flight_id"]

print("Critical columns (Decision 1):", CRITICAL)
print()
critical_nulls = {c: int(df[c].isna().sum()) for c in CRITICAL if c in df.columns}
for col, n in critical_nulls.items():
    pct = 100 * n / len(df)
    flag = "FAIL" if n > 0 else "ok"
    print(f"  {col:18s}  {n:>8,} nulls  ({pct:.4f}%)  [{flag}]")

critical_failed = {c: n for c, n in critical_nulls.items() if n > 0}
if critical_failed:
    result_7c = f"FAIL — nulls in: {critical_failed}"
    # Uncomment to inspect offending rows:
    # for col in critical_failed:
    #     print(f"\nSample of rows with null {col}:")
    #     print(df[df[col].isna()].head().to_string())
else:
    result_7c = "PASS — all critical columns non-null"
print(f"\nResult: {result_7c}")

Critical columns (Decision 1): ['time', 'icao24', 'lat', 'lon', 'baroaltitude', 'flight_id']

  time                       0 nulls  (0.0000%)  [ok]
  icao24                     0 nulls  (0.0000%)  [ok]
  lat                        0 nulls  (0.0000%)  [ok]
  lon                        0 nulls  (0.0000%)  [ok]
  baroaltitude               3 nulls  (0.0002%)  [FAIL]
  flight_id                  0 nulls  (0.0000%)  [ok]

Result: FAIL — nulls in: {'baroaltitude': 3}


### 7d — Duplicate detection (three composite keys)

**What this cell does.** Counts duplicate rows by three different composite keys. The combination of the three counts is **diagnostic** — it tells us not just *whether* duplicates exist, but *what kind*.

**Why three keys instead of one.** ADS-B data has known edge cases that complicate 'what counts as a duplicate.' We discussed three definitions:

1. **`(flight_id, time)`** — *loose*. 'Same flight at the same epoch second has multiple rows.' Catches re-run inserts AND multi-receiver edge cases (where two ground stations picked up the same broadcast and the aggregator didn't fully merge them).
2. **`(flight_id, time, lat, lon)`** — *medium*. 'Same flight, same time, same place.' Filters out the multi-receiver edge case (different receivers see slightly different lat/lon due to GPS jitter). What remains is more suspicious: same observation literally inserted twice.
3. **Full row** — *strict*. 'Every column matches.' Definite pipeline bug — the same row was literally re-inserted byte-for-byte.

**How the three numbers diagnose the issue.**

| Pattern | Diagnosis | Action |
|---|---|---|
| All three = 0 | Clean data | PASS |
| (flight_id, time) > 0; medium = 0; full = 0 | Multi-receiver edge case (legitimate) | REVIEW. Investigate, but probably not a bug. |
| medium > 0; full = 0 | Same place at same instant — suspicious | REVIEW. Either re-broadcast at sub-second cadence or a metadata-mutation bug. |
| full > 0 | Pipeline bug — same row inserted twice | FAIL. Open issue, ask Monica to investigate her export script. |

**Why time granularity matters here.** Monica's `time` is in epoch *seconds* (1-second granularity). Within one second, a flight typically broadcasts once, so same-(flight_id, time) collisions are rare. But they do happen — that's why we report all three counts and interpret combinatorially rather than picking one definition.

**What to look for.**
- All three counts = 0 → PASS.
- Any non-zero → see the diagnosis table above. The result string the cell prints already encodes which case fired.

In [27]:
# Decision 3: three duplicate definitions, reported together for diagnostic interpretation
n_dup_flight_time = int(df.duplicated(subset=["flight_id", "time"]).sum())
n_dup_flight_time_pos = int(df.duplicated(subset=["flight_id", "time", "lat", "lon"]).sum())
n_dup_full = int(df.duplicated().sum())

print(f"Total rows:                                              {len(df):,}")
print(f"Duplicate (flight_id, time) pairs:                       {n_dup_flight_time:,}")
print(f"Duplicate (flight_id, time, lat, lon) tuples:            {n_dup_flight_time_pos:,}")
print(f"Full-row duplicates:                                     {n_dup_full:,}")
print()

# Diagnostic interpretation (see markdown table above)
if n_dup_full > 0:
    result_7d = f"FAIL — {n_dup_full:,} full-row duplicates indicate pipeline re-insertion bug"
elif n_dup_flight_time_pos > 0:
    result_7d = f"REVIEW — {n_dup_flight_time_pos:,} same-position-same-instant duplicates; investigate"
elif n_dup_flight_time > 0:
    result_7d = (
        f"REVIEW — {n_dup_flight_time:,} (flight_id, time) collisions with different positions; "
        "likely multi-receiver edge case, investigate"
    )
else:
    result_7d = "PASS — no duplicates at any granularity"
print(f"Result: {result_7d}")

Total rows:                                              1,774,859
Duplicate (flight_id, time) pairs:                       0
Duplicate (flight_id, time, lat, lon) tuples:            0
Full-row duplicates:                                     0

Result: PASS — no duplicates at any granularity


### 7d.1 — Duplicate diagnostic (run only if 7d FAILed)

**What this cell does.** Characterizes the duplicate rows so we can brief upstream (Monica). Three pieces of information:

1. **Multiplicity distribution** — how many copies each unique row has. "Every row exactly twice" implies a clean overlap; mixed (some 2x, some 3x, some 4x) implies messier re-runs.
2. **Per-day duplication rate** — which calendar dates carry the duplicates. If only certain days are affected, the export ranges overlapped on those days. Uniform across days means the script ran twice over the full range.
3. **Sample row** — pulls the most-repeated row and prints its copies. Confirms the duplicates are byte-for-byte identical (vs. multi-receiver near-duplicates).

**Why this is Phase 2 work, not EDA.** This is diagnostic — it characterizes a *bug* in the upstream pipeline so we can fix it. EDA is for hypothesizing about model-relevant patterns in clean data. Once Monica re-extracts and re-uploads, this snapshot is discarded; nothing learned here biases modeling decisions.

**What to do with the output.** Paste the multiplicity and per-date tables into the GitHub issue / Discord message to Monica. She can then trace the duplication back to her export script (likely overlapping `DIA` ranges, missing `ON CONFLICT DO NOTHING`, or absence of `df.drop_duplicates()` before insert).

In [28]:
n_total = len(df)
n_unique = len(df.drop_duplicates())
n_extras = n_total - n_unique
print(f"Total rows:        {n_total:,}")
print(f"Unique rows:       {n_unique:,}")
print(f"Extra (dup) rows:  {n_extras:,}  ({100 * n_extras / n_total:.2f}% of total)")
print()

# Multiplicity: hash each row, count how often each hash appears
row_hashes = pd.util.hash_pandas_object(df, index=False)
hash_counts = row_hashes.value_counts()
multiplicity = hash_counts.value_counts().sort_index()
print("Multiplicity (how many unique rows appear N times):")
for n, count in multiplicity.items():
    print(f"  {int(n)}x: {int(count):>9,} unique rows  ->  {int(n) * int(count):>10,} total instances")
print()

# Time pattern: are duplicates concentrated on specific dates?
df_check = df.copy()
df_check["_date"] = pd.to_datetime(df_check["time_utc"], utc=True).dt.date
df_check["_is_extra"] = df.duplicated(keep="first")
by_day = df_check.groupby("_date")["_is_extra"].agg(extras="sum", total="count")
by_day["pct_extras"] = (100 * by_day["extras"] / by_day["total"]).round(2)
print("Duplicates by calendar date:")
print(by_day.to_string())
print()

# Sample: find the most-repeated row and show its copies
most_repeated_hash = hash_counts.idxmax()
most_repeated_count = int(hash_counts.max())
matching = df[row_hashes.values == most_repeated_hash].reset_index(drop=True)
print(f"Most-repeated row appears {most_repeated_count} times. First 3 copies:")
cols = ["time", "time_utc", "icao24", "flight_id", "lat", "lon", "baroaltitude", "velocity"]
print(matching[cols].head(3).to_string())

Total rows:        1,774,859
Unique rows:       1,774,859
Extra (dup) rows:  0  (0.00% of total)

Multiplicity (how many unique rows appear N times):
  1x: 1,774,859 unique rows  ->   1,774,859 total instances

Duplicates by calendar date:
            extras   total  pct_extras
_date                                 
2026-03-10       0  338227         0.0
2026-03-11       0  349313         0.0
2026-03-12       0  375002         0.0
2026-03-13       0  375058         0.0
2026-03-14       0  337259         0.0

Most-repeated row appears 1 times. First 3 copies:
         time                   time_utc  icao24          flight_id        lat      lon  baroaltitude    velocity
0  1773116901  2026-03-10T04:28:21+00:00  346318  346318_1773115244  40.666168 -1.16156        8839.2  228.555027


### 7d.2 — Conditional dedup of the snapshot
                                                                                                                                                       
  The default Phase 2 rule is **the snapshot mirrors the source** — don't silently clean. Phase 3 (preprocessing) is where data cleaning belongs.

  But there's an override case for full-row duplicates specifically. Three conditions all need to hold:                                                
                                                                                                                                                       
  1. The duplicates are byte-for-byte identical re-inserts — no information lost by removing them                                                      
  2. There's no preprocessing decision to make — the action is unambiguous: drop them                                                             
  3. Keeping them would weight some flights more heavily in training, distorting the model's normality estimates
                                                                                                                                                       
  When those conditions hold, this cell produces a **second** parquet alongside the raw mirror:
  - The raw is the audit evidence (proof the issue existed at snapshot time)                   
  - The deduped is the canonical version downstream phases load                       
                                                                                                                                                     
  The manifest references the deduped one. `02-data.md` documents both.              
                                                                                                                                                     
  When duplicates are NOT found (the expected case for clean batches from a working extraction script), this cell produces nothing — the raw parquet IS the canonical version, no second file needed.                                                                                                                                              
  This cell is therefore **safe to run on any batch**. It self-detects whether the override applies. For a future batch, an unexpected non-zero removal count is a signal that something upstream regressed — not just a routine cleanup. The cell prints an "ACTION REQUIRED" notice in that case so it can't be silently ignored.                          
                                                            
  Dedup rule (when it fires): drop full-row duplicates. Stricter than composite-key dedup — matches the "byte-for-byte identical" framing from cell 7d.
   
  The key shift in tone vs the earlier markdown: this version emphasizes the cell handles both cases automatically and explicitly flags that an unexpected dedup in a future batch is itself a signal worth investigating.

In [29]:
# Dedup rule: drop full-row duplicates.                                                                                                              
# Only produces a separate deduped parquet if dedup actually removes rows                                                                            
# (the override case described in the markdown above). For clean batches,                                                                            
# the raw parquet IS the canonical version — no second file needed.                                                                                  
                                                                                                                                                       
from datetime import date                                                                                                                            
                                                                                                                                                       
before_n = len(df)                                        
df_deduped = df.drop_duplicates().reset_index(drop=True)
after_n = len(df_deduped)                                                                                                                            
removed = before_n - after_n
removed_pct = 100 * removed / before_n if before_n else 0                                                                                            
                                                                                                                                                       
print(f"Before dedup: {before_n:,} rows")
print(f"After dedup:  {after_n:,} rows")                                                                                                             
print(f"Removed:      {removed:,} ({removed_pct:.2f}%)")                                                                                             
print()
                                                                                                                                                       
if removed == 0:                                                                                                                                     
    print("PASS — no duplicates found.")
    print("The raw parquet IS the canonical version. No deduped file needed.")                                                                       
    deduped_path = None                                                                                                                              
    deduped_hash = None
else:                                                                                                                                                
    # Derive deduped filename from the same SNAPSHOT_NAME, but replace
    # the "__snapshot_" prefix with "__deduped_".                                                                                                    
    deduped_basename = SNAPSHOT_NAME.replace("__snapshot_", "__deduped_")                                                                            
    deduped_path = REPO_ROOT / "data" / "raw" / f"{deduped_basename}.parquet"                                                                        
    save_snapshot_parquet(df_deduped, deduped_path)                                                                                                  
    deduped_hash = compute_file_hash(deduped_path)                                                                                                   
                                                            
    print(f"OVERRIDE APPLIED — deduped parquet produced.")                                                                                           
    print(f"Saved: {deduped_path.relative_to(REPO_ROOT)}")
    print(f"sha256: {deduped_hash}")                                                                                                                 
    print(f"size: {deduped_path.stat().st_size / 1024 / 1024:.2f} MB")
    print()                                                                                                                                          
    print("ACTION REQUIRED:")                             
    print("  1. Upload both raw and deduped parquets to Drive (with .sha256 sidecars)")                                                              
    print("  2. Update manifest.yml > gates.data.dataset_hash with the DEDUPED hash")                                                                
    print("  3. Open a [Bug] issue if dedup was unexpected for this batch")

Before dedup: 1,774,859 rows
After dedup:  1,774,859 rows
Removed:      0 (0.00%)

PASS — no duplicates found.
The raw parquet IS the canonical version. No deduped file needed.


### 7d.3 — Write sha256 sidecar files
                                                                                                                                                       
  For each parquet we just produced (raw, and deduped if it exists), write a `.sha256` sidecar file containing the hash and the parquet filename in the standard `shasum` format.

  **Why sidecars in addition to the manifest hash.** The manifest hash lives in the repo; it's our internal contract. The sidecar lives in Drive *alongside* the parquet; it lets anyone (teammate, reviewer, future-us) verify a parquet's integrity with a single shell command without needing to
  clone the repo:

  ```bash
  shasum -a 256 -c <filename>.sha256

  That's not theoretical — when a teammate pulls a parquet from Drive to run Phase 3 or Phase 4 locally, the sidecar is how they know the file arrived 
  intact. Without it, "is this the canonical version?" turns into a manual lookup against the manifest.
                                                                                                                                                       
  Why this is its own cell rather than baked into earlier cells. The dedup cell may or may not produce a deduped parquet (depends on whether dups were 
  found). The raw parquet is always produced. Centralizing sidecar generation in one cell handles both cases cleanly and keeps the responsibility in
  one place.

In [30]:
# Write sha256 sidecar for raw parquet (always)
raw_path = REPO_ROOT / "data" / "raw" / f"{SNAPSHOT_NAME}.parquet"                                                                                   
raw_hash = compute_file_hash(raw_path)
raw_sidecar = raw_path.with_suffix(".sha256")                                                                                                        
raw_sidecar.write_text(f"{raw_hash}  {raw_path.name}\n")  
print(f"Wrote {raw_sidecar.relative_to(REPO_ROOT)}")                                                                                                 
print(f"  hash: {raw_hash}")                              
                                                                                                                                                       
# Write sha256 sidecar for deduped parquet (only if it exists)
if deduped_path is not None:                                                                                                                         
    deduped_sidecar = deduped_path.with_suffix(".sha256") 
    deduped_sidecar.write_text(f"{deduped_hash}  {deduped_path.name}\n")                                                                             
    print(f"\nWrote {deduped_sidecar.relative_to(REPO_ROOT)}")
    print(f"  hash: {deduped_hash}")                                                                                                                 
else:                                                     
    print("\nNo deduped parquet — sidecar not needed.")                                                                                              
                                                                                                                                                       
# Verify both sidecars are well-formed by parsing them back
print("\nVerification (sidecar contents):")                                                                                                          
print(f"  {raw_sidecar.name}:")                                                                                                                      
print(f"    {raw_sidecar.read_text().strip()}")
if deduped_path is not None:                                                                                                                         
    print(f"  {deduped_sidecar.name}:")                   
    print(f"    {deduped_sidecar.read_text().strip()}")

Wrote data/raw/lemd_20260310_to_20260314__snapshot_2026-05-11.sha256
  hash: 16f1bd2cbdbd519ce7bde6fbbc8df5012b188b54c5598bffc310cef34b0c6899

No deduped parquet — sidecar not needed.

Verification (sidecar contents):
  lemd_20260310_to_20260314__snapshot_2026-05-11.sha256:
    16f1bd2cbdbd519ce7bde6fbbc8df5012b188b54c5598bffc310cef34b0c6899  lemd_20260310_to_20260314__snapshot_2026-05-11.parquet


### 7e — Range bounds + LEMD bbox sanity

**What this cell does.** Two checks combined:
1. Verifies each numeric column's values fall within *generous physical bounds* — bounds chosen to catch broken data (sentinel values, NaN-replaced-with-0, off-by-sign bugs), NOT to filter for drone-shaped trajectories.
2. Cross-checks that lat/lon are inside a ~200km bounding box around LEMD — a soft sanity check on Monica's spatial filter (`MAX_RADIUS_M = 200_000`).

**Why generous bounds, not tight ones.** This is a critical Phase 2 distinction (Decision 4):

- **Phase 2 range check** asks 'is this value broken?' — we use the broadest physically possible bounds, like `lat ∈ [-90, 90]` or `velocity ∈ [0, 400]` (just above Mach 1). Anything outside is impossible (sentinel, sign error, type bug). Anything inside is real ADS-B data, even if not what we want to model.
- **Phase 5 feature filter** asks 'is this row interesting for our model?' — that's where the design doc's filter `alt < 1500m AND vel < 50 m/s` lives. It's NOT a Phase 2 concern.

Mixing the two would mean rejecting valid airliner data as 'broken,' which is wrong.

**What sentinels look like.** Common patterns to spot:
- `min` of `-9999` or `-1` on altitude → 'GPS error' sentinel encoded as a numeric value.
- `min` of `0` on velocity for many rows → could be 'speed unknown' encoded as 0 (vs. a stationary object).
- `max` of `360` for heading is fine; `max` of `360.0001` is float arithmetic, not a violation; `max` of `400` would be a real bug.
- `max` of `dist_to_runway_m > 200_000` would mean Monica's spatial filter let something through.

**Why LEMD bbox cross-check.** Monica filters by `dist_to_runway_m <= 200km` (haversine), but lat/lon should also fall within roughly `[38.5°, 42.5°] × [-6.0°, -1.2°]`. If they don't, either her filter has a bug or there's a row from a non-LEMD airport leaking in. The bbox is a soft check — `< 1%` outside is the threshold for PASS.

**What to look for.**
- `All numeric columns within physical bounds.` and `< 1%` outside bbox → PASS.
- Any violations in physical bounds → REVIEW. Look at the `min_seen` / `max_seen` to diagnose: is it a sentinel value? a sign bug? a type cast issue?
- Bbox > 1% → REVIEW. Either the spatial filter has a bug or rows from another airport are leaking in.

In [31]:
# Decision 4: generous physical bounds. Phase 2 catches BROKEN data, not OUT-OF-INTEREST.
# Drone-shape filtering (alt < 1500m, vel < 50 m/s) belongs in Phase 5 (features).
PHYSICAL_BOUNDS = {
    "lat": (-90, 90),
    "lon": (-180, 180),
    "baroaltitude": (-500, 50_000),       # m; -500 covers Dead Sea elevation, 50k is 2x airliner cruise
    "geoaltitude": (-500, 50_000),
    "velocity": (0, 400),                   # m/s; Mach 1 ~340, this is buffer
    "heading": (0, 360),
    "vertrate": (-100, 100),                # m/s; aerobatics ~50, this is generous
    "velocity_kmh": (0, 1500),
    "dist_to_runway_m": (0, 200_000),       # MAX_RADIUS_M from Monica's filter
}

violations = {}
for col, (lo, hi) in PHYSICAL_BOUNDS.items():
    if col not in df.columns:
        continue
    s = df[col].dropna()
    if len(s) == 0:
        continue
    out_lo = int((s < lo).sum())
    out_hi = int((s > hi).sum())
    if out_lo or out_hi:
        violations[col] = {
            "below_min": out_lo,
            "above_max": out_hi,
            "min_seen": float(s.min()),
            "max_seen": float(s.max()),
            "expected": (lo, hi),
        }

if violations:
    print("Physical-bounds violations:")
    for col, info in violations.items():
        print(f"  {col}: {info}")
    physical_result = f"REVIEW — {len(violations)} columns with violations"
else:
    print("All numeric columns within physical bounds.")
    physical_result = "PASS"

# LEMD bbox sanity cross-check (~200km box around LEMD ~ 40.47N, -3.56W)
LEMD_BBOX = {"lat": (38.5, 42.5), "lon": (-6.0, -1.1)}  # widened 2026-05-11 per Known issue #6 in 02-data.md
out_of_bbox = (
    (df["lat"] < LEMD_BBOX["lat"][0]) | (df["lat"] > LEMD_BBOX["lat"][1])
    | (df["lon"] < LEMD_BBOX["lon"][0]) | (df["lon"] > LEMD_BBOX["lon"][1])
).sum()
out_of_bbox_pct = 100 * out_of_bbox / len(df)
print(f"\nLEMD bbox sanity: {out_of_bbox:,} rows outside ~200km box ({out_of_bbox_pct:.4f}%)")
bbox_result = "PASS" if out_of_bbox_pct < 1.0 else f"REVIEW — {out_of_bbox_pct:.2f}% outside expected geography"

result_7e = f"physical={physical_result}; bbox={bbox_result}"
print(f"\nResult: {result_7e}")

All numeric columns within physical bounds.

LEMD bbox sanity: 0 rows outside ~200km box (0.0000%)

Result: physical=PASS; bbox=PASS


## 8 — Pipeline consistency check

**What this cell does.** Re-derives four pipeline-computed columns from the raw inputs and compares to Monica's stored values. The four columns are `velocity_kmh`, `dist_to_runway_m`, `flight_phase`, and `time_utc`. If any disagree beyond a tight tolerance, that's a sign Monica's deployed pipeline diverges from the version in the repo (version skew, code drift, environment mismatch).

**Why this matters.** Monica's pipeline writes derived columns to Supabase based on a particular version of `backend/research/src/sadar_research/trajectory_anomaly/data/opensky.py`. If she deployed an older or newer version than what's in the repo, her stored values won't match what we re-compute *from this repo's code*. This is a real risk in collaborative projects — and the consistency check catches it cheaply.

**Why tight tolerances (Decision 5).** The whole point is to *detect* drift; loose tolerances would mask it. Tolerance choices reflect each column's mathematical structure:

- **`velocity_kmh = velocity × 3.6`** — pure float multiplication. The only difference possible is IEEE-754 rounding noise. Tolerance: `< 1e-6` km/h.
- **`dist_to_runway_m`** — haversine distance over 8 runway-threshold coordinates, then min. Trig functions (sin, cos, arctan2) accumulate small float errors, so the tolerance is slightly looser. Tolerance: `< 1.0` m.
- **`time_utc = pd.to_datetime(time, unit='s', utc=True)`** — epoch-to-datetime is deterministic. Tolerance: `< 1.0` s (effectively exact).
- **`flight_phase`** — discrete output of a deterministic rule. *No float noise possible.* Tolerance: 100% exact match.

**What `flight_phase` exact-match catches.** Since the rule is deterministic on its inputs (`onground`, `baroaltitude`, `vertrate`, `dist_to_runway_m`), any disagreement means *either* Monica's stored phase or our re-derived phase is wrong. Both should be the same code path. A < 100% match here is the most likely sign of code drift.

**What to look for.**
- All four diffs within tolerance → PASS.
- `flight_phase agreement = 99.X%` → REVIEW. Some rows disagree. Most likely Monica's deployed code differs from the repo. Check `git log backend/research/src/sadar_research/trajectory_anomaly/data/opensky.py` and ask Monica which version she ran.
- Large numeric diffs → REVIEW. Could be a different formula, different runway coordinates, different unit assumptions.

**What REVIEW triggers.** Investigate before continuing. The fix is usually 'Monica re-runs her export script with the latest repo version' — but we want to confirm the divergence first.

In [32]:
# Decision 5: tight tolerances. Catches version skew or upstream drift.
TOL_KMH = 1e-6        # km/h; pure multiplication, only IEEE-754 rounding
TOL_DIST = 1.0        # meters; haversine accumulates small float noise
TOL_TIME = 1.0        # seconds; epoch-to-datetime should be effectively exact
TOL_PHASE = 1.0       # 100% match; rule is deterministic, no float noise possible

# 1) velocity_kmh = velocity * 3.6
re_kmh = df["velocity"] * 3.6
diff_kmh = (df["velocity_kmh"] - re_kmh).abs().dropna()
max_kmh_diff = float(diff_kmh.max()) if len(diff_kmh) else 0.0
print(f"velocity_kmh         max abs diff = {max_kmh_diff:.6e}    (tol {TOL_KMH:.0e})")

# 2) dist_to_runway_m = haversine to closest of 8 LEMD runway thresholds
re_dist = distance_to_closest_runway(df["lat"], df["lon"])
diff_dist = (df["dist_to_runway_m"] - re_dist).abs().dropna()
max_dist_diff = float(diff_dist.max()) if len(diff_dist) else 0.0
print(f"dist_to_runway_m     max abs diff = {max_dist_diff:.6e} m  (tol {TOL_DIST} m)")

# 3) flight_phase from rule on (onground, baroaltitude, vertrate, dist_to_runway_m)
work = df[["onground", "baroaltitude", "vertrate", "dist_to_runway_m"]].copy()
re_phase = pd.Series(calculate_flight_phase(work), index=df.index)
phase_match = float((df["flight_phase"].fillna("") == re_phase.fillna("")).mean())
print(f"flight_phase         agreement   = {100 * phase_match:.4f}%  (tol 100%)")

# 4) time_utc = pd.to_datetime(time, unit='s', utc=True)
re_time = pd.to_datetime(df["time"], unit="s", utc=True)
mon_time = pd.to_datetime(df["time_utc"], utc=True, errors="coerce")
diff_seconds = (mon_time - re_time).abs().dt.total_seconds().dropna()
max_time_diff = float(diff_seconds.max()) if len(diff_seconds) else 0.0
print(f"time_utc             max abs diff = {max_time_diff:.6f} s  (tol {TOL_TIME} s)")

consistency_passed = (
    max_kmh_diff < TOL_KMH
    and max_dist_diff < TOL_DIST
    and phase_match >= TOL_PHASE
    and max_time_diff < TOL_TIME
)
result_8 = "PASS" if consistency_passed else "REVIEW — drift detected, investigate"
print(f"\nResult: {result_8}")

velocity_kmh         max abs diff = 6.821210e-12    (tol 1e-06)
dist_to_runway_m     max abs diff = 6.082701e-09 m  (tol 1.0 m)
flight_phase         agreement   = 100.0000%  (tol 100%)
time_utc             max abs diff = 0.000000 s  (tol 1.0 s)

Result: PASS


## 9 — Distribution sanity

**What this cell does.** Saves a histogram of each numeric column to `docs/research/trajectory-anomaly/figures/02-data/` for inclusion in the writeup. *This is sanity-only* — looking for sentinels and shape problems, not doing real exploratory analysis.

**Why we look at distributions at Phase 2 at all.** Some data issues only show up visually:
- A spike at `-9999` or `-1` → sentinel value (the formal range check might catch it, but a histogram surfaces it visibly)
- A bimodal altitude distribution where you'd expect unimodal → mixing two populations (e.g., commercial cruise + ground-level)
- A perfect peak at zero where you'd expect a distribution → coercion of nulls to zero (a real bug)
- A sharp cutoff at a round number (e.g., velocity capped at 250) → upstream filter applied without us knowing

**Why this is NOT Phase 4 EDA.** Phase 4 is where we look for *interesting* patterns and form modeling hypotheses. Phase 2 just checks that the data isn't *broken*. If you find yourself thinking 'oh interesting, the altitude distribution looks like X' — note it for Phase 4 and move on; don't analyze it now.

**What to look for in the saved figures.**
- Each histogram should look 'physical' — smooth-ish, no implausible spikes, no clipped flat regions.
- Sentinel spike at a single weird value → flag in `02-data.md`.
- Multi-modal where unexpected → flag for Phase 4 (could be informative).
- Empty plot → that column is all nulls or has a single value. Investigate.

**File naming.** Each figure is saved as `{TABLE}__{column}.png` so multiple days don't overwrite each other if you re-run with different `TABLE` values.

In [33]:
NUMERIC_PLOT = [
    "lat", "lon", "baroaltitude", "geoaltitude",
    "velocity", "heading", "vertrate",
    "velocity_kmh", "dist_to_runway_m",
]

for col in NUMERIC_PLOT:
    if col not in df.columns:
        continue
    fig, ax = plt.subplots(figsize=(8, 4))
    df[col].dropna().hist(bins=60, ax=ax)
    ax.set_title(f"{SNAPSHOT_NAME} — {col} (n={int(df[col].notna().sum())})")
    ax.set_xlabel(col)
    ax.set_ylabel("count")
    fig_path = FIG_DIR / f"{SNAPSHOT_NAME}__{col}.png"
    fig.tight_layout()
    fig.savefig(fig_path, dpi=120)
    plt.close(fig)
    print(f"saved {fig_path.relative_to(REPO_ROOT)}")

saved docs/research/trajectory-anomaly/figures/02-data/lemd_20260310_to_20260314__snapshot_2026-05-11__lat.png
saved docs/research/trajectory-anomaly/figures/02-data/lemd_20260310_to_20260314__snapshot_2026-05-11__lon.png
saved docs/research/trajectory-anomaly/figures/02-data/lemd_20260310_to_20260314__snapshot_2026-05-11__baroaltitude.png
saved docs/research/trajectory-anomaly/figures/02-data/lemd_20260310_to_20260314__snapshot_2026-05-11__geoaltitude.png
saved docs/research/trajectory-anomaly/figures/02-data/lemd_20260310_to_20260314__snapshot_2026-05-11__velocity.png
saved docs/research/trajectory-anomaly/figures/02-data/lemd_20260310_to_20260314__snapshot_2026-05-11__heading.png
saved docs/research/trajectory-anomaly/figures/02-data/lemd_20260310_to_20260314__snapshot_2026-05-11__vertrate.png
saved docs/research/trajectory-anomaly/figures/02-data/lemd_20260310_to_20260314__snapshot_2026-05-11__velocity_kmh.png
saved docs/research/trajectory-anomaly/figures/02-data/lemd_20260310_to_

## 10 — Class balance

**What this cell does.** Counts the values in `operation` (arrival / departure / unknown) and `flight_phase` (the rule-derived classification). Reports unique-flight counts per operation as well.

**Why it matters.** Class balance is relevant for *downstream awareness*, even though we don't act on it in Phase 2:
- A heavily imbalanced `operation` distribution (e.g., 90% arrivals, 5% departures, 5% unknown) suggests Monica's day captured one direction more than the other — informative for Phase 4 EDA and for the Phase 6 split design.
- The `flight_phase` distribution tells us how trajectories are spread across the rule's six classes. For LSTM training (Phase 6), we want diverse phase coverage — if 95% of rows are 'cruise,' the model won't learn the dynamics of takeoff or approach.
- Many `operation = unknown` rows mean Monica's Trino query couldn't classify them (the airport-of-origin/destination wasn't 'LEMD' for either side). That's a known case for through-traffic; flag it for the writeup.

**Why we don't act on imbalance now.** Phase 2 just records what's there. Phase 5 (features) and Phase 6 (split design + class weighting) decide how to handle imbalance.

**What to look for.**
- `operation`: roughly balanced arrival vs departure if the day was symmetric. Heavy unknown count means many through-flights or unclassified.
- `flight_phase`: all six classes (`on_ground`, `takeoff`, `climb`, `cruise`, `approach`, `descent`) should be represented. A missing class is suspicious.
- Unique flights per operation: should roughly match the row counts ratio. A wildly different ratio means very long arrivals vs short departures (or vice versa) — interesting, but not a Phase 2 issue.

In [34]:
print("operation:")
print(df["operation"].value_counts(dropna=False).to_string())
print("\nflight_phase:")
print(df["flight_phase"].value_counts(dropna=False).to_string())
print("\nUnique flights per operation:")
print(df.groupby("operation")["flight_id"].nunique().to_string())

operation:


operation
arrival      1093846
departure     681013

flight_phase:
flight_phase
descent      811007
climb        421982
cruise       340411
approach     125335
takeoff       76122
on_ground         2

Unique flights per operation:
operation
arrival      725
departure    701


## 11 — Volume metrics (cumulative across batches)

  **What this cell does.** Globs for all parquets on disk under `data/raw/`. For each batch (identified by everything before the `__snapshot_` or `__deduped_` suffix), picks the canonical version — preferring the deduped parquet, falling back to the raw if no dedup was applied for that batch.
  Concatenates everything and reports cumulative metrics that feed Decision 7 (the Enough verdict).
                                                            
  **Why disk-glob rather than `available`.** The discovery cell (3) returns Supabase table names — they describe what's *currently in Supabase*. Volume metrics need to read from *what's been validated and saved across all cycles*. Different scopes. Globbing the disk reads our durable record, not Supabase's transient state. As Monica's cycles accumulate, this cell automatically picks up the new parquets without any rewiring.

  **Why prefer deduped over raw per batch.** When dedup was applied (Response B), the deduped parquet is the canonical version Phase 3+ will load. The raw is audit evidence only. The volume metrics should reflect what will actually be trained on.

  **Why diversity matters more than raw count.** ADS-B data has a structural correlation problem: consecutive timesteps within one flight are highly dependent on each other. The 'effective' sample size for a trajectory model is much closer to the *trajectory count* than the *row count*. A million state vectors from one day teaches the model 'one Tuesday morning at LEMD,' not 'normal flight at LEMD.'

  **Why coverage matters.** The model needs to see different times of day, days of week, weather conditions, and (subtly) runway configurations. Even just hitting all 7 days of week and a wide range of hours starts to give the model the kind of variation it needs to generalize.
  
  **Three thresholds (locked in Decision 7).**

  1. **500 trajectories** — *project viability floor* (from design doc). Below this, the design doc says switch approaches.
  2. **5,000 trajectories AND 30 calendar days** — *trainable*. Below this, develop the pipeline but don't trust trained models yet.
  3. **20,000 trajectories AND 90 calendar days** — *reliable*. Above this, the model has diverse normality estimates.

  **What to look for.**

  - `Batches available`: how many parquet batches the disk-glob found. Each Monica cycle = 1 batch.
  - `Calendar days seen`: distinct calendar dates spanned across all batches. Maps to the threshold buckets.
  - `Trajectories total`: unique `flight_id` values across all batches. Maps to the threshold buckets.
  - `Time range`: earliest to latest state vector across batches. Wider = more diversity.
  - `Day-of-week coverage`: how many of [Mon, Tue, Wed, Thu, Fri, Sat, Sun] are represented. Less than 7 means biased coverage.
  - `Hour-of-day coverage`: how many of the 24 hours are represented across the dataset.

In [35]:
                                                                                                                                                     
# Aggregate across all parquets on disk for the Enough verdict (Decision 7).
# Prefer the deduped parquet for each batch (canonical); fall back to raw
# snapshot if no deduped exists for a batch.                                                                                                         
   
def batch_id(path):                                                                                                                                  
    """Strip __snapshot_<date> or __deduped_<date> suffix to get batch identity."""
    name = path.stem                                                                                                                                 
    if "__snapshot_" in name:
        return name.split("__snapshot_")[0]                                                                                                          
    if "__deduped_" in name:                              
        return name.split("__deduped_")[0]                                                                                                           
    return name                                           

raw_files = sorted((REPO_ROOT / "data" / "raw").glob("lemd_*__snapshot_*.parquet"))                                                                  
deduped_files = sorted((REPO_ROOT / "data" / "raw").glob("lemd_*__deduped_*.parquet"))
                                                                                                                                                       
deduped_by_batch = {batch_id(p): p for p in deduped_files}
raw_by_batch = {batch_id(p): p for p in raw_files}                                                                                                   
                                                                                                                                                       
# For each batch, prefer deduped, fall back to raw
canonical = []                                                                                                                                       
for bid in sorted(set(raw_by_batch) | set(deduped_by_batch)):
    if bid in deduped_by_batch:                                                                                                                      
        canonical.append(("deduped", deduped_by_batch[bid]))
    elif bid in raw_by_batch:                                                                                                                        
        canonical.append(("snapshot", raw_by_batch[bid])) 
                                                                                                                                                       
print(f"Found {len(canonical)} canonical batch(es):")
for kind, path in canonical:                                                                                                                         
    print(f"  [{kind:>8s}] {path.name}")                  
print()                                                                                                                                              
   
# Load and concat                                                                                                                                    
all_dfs = []                                              
for kind, path in canonical:
    df_d = pd.read_parquet(path, engine="pyarrow")
    df_d["_batch"] = path.stem                                                                                                                       
    all_dfs.append(df_d)
                                                                                                                                                       
df_all = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()                                                                        
   
# Compute metrics                                                                                                                                    
n_batches = len(canonical)                                
n_trajectories = int(df_all["flight_id"].nunique()) if len(df_all) else 0
n_rows_total = int(len(df_all))                                                                                                                      
   
if len(df_all):                                                                                                                                      
    df_all["time_utc_dt"] = pd.to_datetime(df_all["time_utc"], utc=True, errors="coerce")
    time_min = df_all["time_utc_dt"].min()                                                                                                           
    time_max = df_all["time_utc_dt"].max()
    days_seen = sorted(df_all["time_utc_dt"].dt.date.unique())                                                                                       
    dow_seen = sorted({d.weekday() for d in days_seen})  # 0=Mon, 6=Sun
    hours_seen = sorted(df_all["time_utc_dt"].dt.hour.unique().tolist())                                                                             
    n_calendar_days = len(days_seen)                      
else:                                                                                                                                                
    time_min = time_max = None                            
    days_seen = []                                                                                                                                   
    dow_seen = []                                         
    hours_seen = []
    n_calendar_days = 0
                                                                                                                                                       
print("=" * 60)
print("Volume metrics across all canonical batches:")                                                                                                
print("=" * 60)                                           
print(f"  Batches available:        {n_batches}")
print(f"  Calendar days seen:       {n_calendar_days}")                                                                                              
print(f"  Trajectories total:       {n_trajectories:,}")
print(f"  Rows total (canonical):   {n_rows_total:,}")                                                                                               
print(f"  Time range:               {time_min}  →  {time_max}")                                                                                      
print(f"  Day-of-week coverage:     {dow_seen}  (out of [0..6], 0=Mon)")
print(f"  Hour-of-day coverage:     {len(hours_seen)} distinct hours")

Found 2 canonical batch(es):
  [ deduped] lemd_20250310_to_20250314__deduped_2026-05-10.parquet
  [snapshot] lemd_20260310_to_20260314__snapshot_2026-05-11.parquet

Volume metrics across all canonical batches:
  Batches available:        2
  Calendar days seen:       10
  Trajectories total:       2,711
  Rows total (canonical):   2,921,090
  Time range:               2025-03-10 00:58:21+00:00  →  2026-03-14 23:45:42+00:00
  Day-of-week coverage:     [0, 1, 2, 3, 4, 5]  (out of [0..6], 0=Mon)
  Hour-of-day coverage:     24 distinct hours


## 12 — Verdict (Real / Usable / Enough)

**What this cell does.** Produces the three Phase 2 verdict outputs by combining the check results above. The verdict is the final synthesis of everything we've measured.

**Verdict mappings (Decision 6 + Decision 7).**

**Real?** — *Is this actually ADS-B data near LEMD?*
- PASS if schema match (7a) is PASS, no physical-bounds violations (7e), AND ≥99% of rows fall in LEMD bbox (7e).
- FAIL if any of those conditions break.

**Usable?** — *Is the data structurally sound enough to build on?*
- PASS if no critical-column nulls (7c), no full-row duplicates (7d), AND pipeline consistency holds (8).
- FAIL if any of those break.

**Enough?** — *Do we have enough volume + diversity to train an LSTM autoencoder?* (4 buckets):

| Bucket | Condition | What it means |
|---|---|---|
| **NOT_YET** | < 500 trajectories | Below project-viability floor. Per design doc, widen bbox / extend time range / relax altitude filter, or switch approaches. |
| **SOFT_DEV** | 500–4,999 trajectories OR < 30 days | Project is viable. Pipeline development can proceed. DL training requires expansion (≥30 days AND ≥5K trajectories) before Phase 6. |
| **CONDITIONAL** | 5K–19,999 trajectories AND 30–89 days | Trainable; model will learn but with limited diversity. Document the limitation in the writeup. |
| **PASS** | ≥20K trajectories AND ≥90 days | Well-supported; diverse enough for reliable normality estimates. |

**What 'Enough = NOT_YET' actually triggers.** It's NOT a Phase 2 fail — we can still close Phase 2 successfully. But the manifest gets a flag noting volume is below target. Phase 3 and Phase 4 work proceeds. Phase 6 (training) becomes contingent on getting more data first.

**The most likely current state.** If only one day exists (`lemd_2025_03_11`), expected verdict is:
- Real: PASS
- Usable: PASS  
- Enough: SOFT_DEV (≥500 trajectories likely, but <30 days)

**What to do with the verdict.**
- Copy the verdict block into `02-data.md > Verdict`.
- If Enough is NOT_YET or SOFT_DEV, open a follow-up issue: 'Phase 2 follow-up: expand data coverage to ≥30 days.' Assigned to Monica (or whoever picks up the export-script extension).
- Update the manifest's `gates.data.summary` to reflect the verdict outcome.

In [36]:
# Decision 6: Real and Usable mappings
real_pass = (
    result_7a == "PASS"
    and not violations
    and out_of_bbox_pct < 1.0
)
real_verdict = "PASS" if real_pass else "FAIL — review check outputs above"

usable_pass = (
    not critical_failed
    and n_dup_full == 0
    and consistency_passed
)
usable_verdict = "PASS" if usable_pass else "FAIL — review check outputs above"

# Decision 7: Enough — 4 buckets based on trajectory count + day count
if n_trajectories < 500:
    enough_bucket = "NOT_YET"
    enough_action = (
        "Below project-viability floor (<500 tracks). Per design doc: widen bbox, "
        "extend time range, or relax altitude filter."
    )
elif n_calendar_days < 30 or n_trajectories < 5_000:
    enough_bucket = "SOFT_DEV"
    enough_action = (
        "Project viable; pipeline development can proceed. DL training requires "
        "expansion (>=30 days AND >=5K trajectories) before Phase 6."
    )
elif n_calendar_days < 90 or n_trajectories < 20_000:
    enough_bucket = "CONDITIONAL"
    enough_action = (
        "Trainable; model will learn but with limited diversity. Acceptable for "
        "course timeline; document limitation in writeup."
    )
else:
    enough_bucket = "PASS"
    enough_action = "Well-supported; diverse enough for reliable normality estimates."

print("=" * 70)
print(f"Real:    {real_verdict}")
print(f"Usable:  {usable_verdict}")
print(f"Enough:  {enough_bucket}")
print("=" * 70)
print(f"Trajectories: {n_trajectories:,}  (viability floor: 500;  trainable: 5,000;  reliable: 20,000)")
print(f"Days:         {n_calendar_days}             (development OK: <30;  CONDITIONAL: 30-90;  PASS: >=90)")
print(f"Action:       {enough_action}")
print("=" * 70)

Real:    PASS
Usable:  FAIL — review check outputs above
Enough:  SOFT_DEV
Trajectories: 2,711  (viability floor: 500;  trainable: 5,000;  reliable: 20,000)
Days:         10             (development OK: <30;  CONDITIONAL: 30-90;  PASS: >=90)
Action:       Project viable; pipeline development can proceed. DL training requires expansion (>=30 days AND >=5K trajectories) before Phase 6.


## 13 — Next steps

**What this notebook produced.**
- Per-day parquet snapshots in `data/raw/` (gitignored, but hashes recorded).
- Per-day sha256 hashes (printed in cell 4).
- Histograms in `docs/research/trajectory-anomaly/figures/02-data/`.
- Validation outcomes (cells 7a–7e, 8) — six values total.
- Class balance and volume metrics (cells 10–11).
- Verdict (cell 12).

**What to do with these outputs to close Phase 2.**

1. **Write `docs/research/trajectory-anomaly/lifecycle/02-data.md`** following the Phase 2 template (`/ml-lifecycle/references/docs-template.md`). Sections to fill:
   - `## Source(s)` — Supabase project URL + tables, OpenSky terms (research account)
   - `## Schema (data dictionary)` — paste `schema_df` from cell 6
   - `## Volume` — paste cell 11 output
   - `## Validation report` — paste outcomes from cells 7a–7e, 8
   - `## Known issues` — for any FAIL or REVIEW outcome above

2. **Update `docs/research/trajectory-anomaly/lifecycle/manifest.yml`**:
   - `gates.data.status: passed`
   - `gates.data.passed_at: 2026-05-08`
   - `gates.data.artifact: docs/research/trajectory-anomaly/lifecycle/02-data.md`
   - `gates.data.dataset_hash:` paste the per-day hashes from cell 4 as a dict (one entry per `lemd_*` day)
   - `gates.data.summary:` one-line summary of the verdict (e.g., `'1 day, 1488 trajectories. Real=PASS Usable=PASS Enough=SOFT_DEV. Volume expansion required before Phase 6.'`)
   - `current_phase: preprocess`

3. **Open follow-up issues** for any FAIL / REVIEW (per Decision 2):
   - Title pattern: `[Bug]: <specific issue>` for failures
   - Title pattern: `[Task]: Phase 2 follow-up — expand data coverage` for Enough = NOT_YET / SOFT_DEV
   - Tag the team (Monica + everyone)

4. **Commit the validation outputs and update the PR**:
   - Commit `02-data.md`, the updated `manifest.yml`, and the figures
   - Push to branch `12-task-phase2-data-validation`
   - The existing PR auto-updates

5. **Run `/develop` Build Step 3** (`/ship`) to finalize the PR for review.